# L-CAD baseline on the Phase 0 smoke subset (Colab)

L-CAD (NeurIPS 2023, language-based colorization with a diffusion prior) is the **language baseline** for chroma-reasoner Phase 0. It does not fit in 4 GB VRAM, so it runs here on a Colab GPU (T4/A100 both fine).

Flow: clone repos → install deps → download weights (Google Drive) → regenerate the deterministic COCO smoke subset → run inference with COCO captions → zip outputs for local evaluation with `scripts/evaluate.py`.

> **Runtime → Change runtime type → GPU** before running.

In [ ]:
!nvidia-smi
import torch; print(torch.__version__, torch.cuda.is_available())

## 1. Clone repos

In [ ]:
!git clone https://github.com/changzheng123/L-CAD.git
!git clone https://github.com/tomqi6195/chroma-reasoner.git

## 2. Install dependencies

In [ ]:
%cd /content
# Colab runtimes are ephemeral: re-run the clone cell after any reset.
# These guards make this cell safe to re-run either way.
!test -d L-CAD || git clone https://github.com/changzheng123/L-CAD.git

# Do NOT `pip install -r L-CAD/requirements.txt`: it is a conda-env dump that
# pins torch==1.12.1, numpy from a local file:// path, and an editable local
# segment-anything — it breaks on Colab and would clobber Colab's CUDA torch.
# Install only what the LDM-style codebase imports, keeping Colab's torch.
# (torchviz: a debugging leftover imported by L-CAD's ldm/models/diffusion/ddim.py.)
!pip install -q pytorch-lightning==1.9.5 omegaconf einops kornia open-clip-torch taming-transformers-rom1504 torchviz
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q gdown pycocotools clean-fid
!pip install -q -e chroma-reasoner

## 3. Download pretrained weights

Weights folder (from the L-CAD README): https://drive.google.com/drive/folders/1_zVJrp_UkFDaZpcC8aLzpv4iPsHADQU-

In [ ]:
import os
!mkdir -p L-CAD/models

CKPT = 'L-CAD/models/coco_weight.ckpt'
# File id of coco_weight.ckpt inside the shared folder (from gdown's listing).
# Only this one file is needed: the config is self-contained (enhanced decoder
# included in the ckpt), so skip the other ~20 GB.
COCO_WEIGHT_ID = '1IFHCIJC_mOWaWwAWOyzXP4_0GchNQRwK'

# Attempt 1: direct download of the single file (8.5 GB).
if not os.path.exists(CKPT):
    !gdown {COCO_WEIGHT_ID} -O {CKPT}

# Attempt 2: Drive's download quota blocks popular files ("many accesses").
# coco_weight.ckpt -> Make a copy (it lands in My Drive; server-side, quota-free).
if not os.path.exists(CKPT):
    from google.colab import drive
    import shutil
    drive.mount('/content/drive')
    shutil.copy('/content/drive/MyDrive/Datasets/coco_weight.ckpt', CKPT)

!ls -lh L-CAD/models   # coco_weight.ckpt must be ~8.5G, not smaller

## 4. Regenerate the smoke subset (deterministic: n=300, seed=0)

Same selection code as local, so stems match for evaluation.

In [ ]:
!cd chroma-reasoner && python scripts/download_coco_subset.py --n 300 --seed 0 --root data/coco

## 5. Run L-CAD inference

Resolved mechanism (from reading the L-CAD source): `python colorization_main.py -m`
(multicolor, no SAM) loads images from `L-CAD/example/` and captions from
`L-CAD/example/test-pair.json`, a **list of `[filename, caption]` pairs**
(`MyDataset(img_dir='example', caption_dir='example', split='test')`).
The checkpoint path is hardcoded as a placeholder (`.models/xxxxx.ckpt`)
in `colorization_main.py` and must be patched — the cell below does it
with `sed`. Outputs land in `L-CAD/image_log/test_{timestamp}_ug_4.5/`
(DDIM 50 steps, guidance 4.5) with the prompt appended to each filename;
the collect cell renames them to `{stem}.png` for evaluation.

In [ ]:
import json
manifest = json.load(open('chroma-reasoner/data/coco/manifest.json'))
pairs = [(im['file_name'], im['captions'][0]) for im in manifest['images'] if im['captions']]
print(len(pairs), 'image/caption pairs; first:', pairs[0])

In [ ]:
import json, glob, os, shutil

# Stage inputs the way L-CAD's demo branch expects:
# example/<file_name> + example/test-pair.json = list of [filename, caption].
# Feeding color images is leak-free: the model only consumes the L channel.
src = 'chroma-reasoner/data/coco/val2017_subset'
for fn, _ in pairs:
    shutil.copy(os.path.join(src, fn), os.path.join('L-CAD/example', fn))
with open('L-CAD/example/test-pair.json', 'w') as f:
    json.dump(pairs, f, indent=2)

# colorization_main.py hardcodes resume_path='.models/xxxxx.ckpt' (a placeholder).
# Of the three released weights, coco_weight.ckpt is the caption-conditioned
# model trained on Extended COCO-Stuff (our setting); multi_weight is the
# instance-aware/SAM variant (inference.py) and auto_weight the no-caption one.
print(glob.glob('L-CAD/models/**/*.ckpt', recursive=True))
!sed -i "s|resume_path *= *'[^']*'|resume_path='models/coco_weight.ckpt'|" L-CAD/colorization_main.py
!grep -n "resume_path" L-CAD/colorization_main.py

# L-CAD also hardcodes a local CLIP path from the author's machine; point it
# at the HF hub id (the text encoder downloads ~700 MB on first use).
!grep -rln "/data/pretrained/clip-vit-large-patch14" L-CAD | xargs -r sed -i "s|/data/pretrained/clip-vit-large-patch14/*|openai/clip-vit-large-patch14|g"

In [ ]:
import re, subprocess

# Verify L-CAD's import chain, installing any modules it still misses
# (module name != pip name for a few common ones).
PIP_NAME = {'cv2': 'opencv-python', 'skimage': 'scikit-image', 'PIL': 'pillow',
            'clip': 'git+https://github.com/openai/CLIP.git'}

for _ in range(10):
    r = subprocess.run(['python', '-c', 'import sys; sys.path.insert(0,"."); import cldm.cldm'],
                       cwd='L-CAD', capture_output=True, text=True)
    if r.returncode == 0:
        print('imports OK'); break
    m = re.search(r"No module named '([\w\.]+)'", r.stderr)
    if not m:
        print(r.stderr[-3000:]); break   # a different error: needs a real look
    pkg = m.group(1).split('.')[0]
    print('installing', PIP_NAME.get(pkg, pkg))
    subprocess.run(['pip', 'install', '-q', PIP_NAME.get(pkg, pkg)])

In [ ]:
# The ckpt was saved with transformers 4.x CLIP key layout
# (cond_stage_model.transformer.text_model.*); transformers 5.x flattened it,
# so a strict load fails with hundreds of missing/unexpected keys. Downgrade,
# and load non-strict to absorb the one residual key (position_ids buffer,
# dropped in 4.31+, harmless).
!pip install -q "transformers<5"

p = 'L-CAD/colorization_main.py'
s = open(p).read()
old = "model.load_state_dict(load_state_dict(resume_path, location='cpu'))"
new = ("m_, u_ = model.load_state_dict(load_state_dict(resume_path, location='cpu'), strict=False); "
       "print('missing:', len(m_), 'unexpected:', len(u_)); "
       "assert len(m_) == 0, ('real weights missing - wrong transformers version?', m_[:5])")
if old in s:
    open(p, 'w').write(s.replace(old, new))
# Healthy output at inference start: "missing: 0 unexpected: 1"
!grep -n "strict=False" L-CAD/colorization_main.py

In [ ]:
# 50-step DDIM x 300 images: roughly 2-4 h on T4, ~40 min on A100.
# For a quick first pass, rewrite test-pair.json with pairs[:20] above and re-run.
# The grep must show coco_weight.ckpt; if not, re-run the staging cell above.
!grep -n resume_path L-CAD/colorization_main.py
!cd L-CAD && python colorization_main.py -m

In [ ]:
import glob, os, re, shutil

# L-CAD saves to L-CAD/image_log/test_{timestamp}_ug_4.5/ and appends the
# prompt to filenames; rename back to {stem}.png so evaluate.py can pair
# outputs with ground truth. COCO stems are all digits.
runs = sorted(glob.glob('L-CAD/image_log/test_*'), key=os.path.getmtime)
out_dir = 'results/lcad'; os.makedirs(out_dir, exist_ok=True)
n = 0
for p in glob.glob(os.path.join(runs[-1], '*.png')):
    stem = re.match(r'\d+', os.path.basename(p)).group(0)
    shutil.copy(p, os.path.join(out_dir, stem + '.png')); n += 1
print(n, 'outputs ->', out_dir)

## 6. Export outputs for local evaluation

In [ ]:
!zip -rq lcad_outputs.zip results/lcad
from google.colab import files
files.download('lcad_outputs.zip')
# Locally: python scripts/evaluate.py --pred results/lcad --gt data/coco/val2017_subset \
#              --manifest data/coco/manifest.json --out results/phase0/lcad